# Session1_Task2_short — Data Cleaning & Transformation

In [1]:
import pandas as pd, numpy as np, warnings; warnings.filterwarnings('ignore')

c = pd.read_csv('customers.csv')
s = pd.read_csv('sales_transactions.csv')

# Missing values
c['age']          = c['age'].fillna(c['age'].median())                         # เติม median
c['phone_number'] = (c['phone_number'].fillna('0').astype(str)
                     .str.replace(r'[^0-9+]','',regex=True).replace(['','00'],'0'))  # ลบอักขระแปลก
s['promotion_id'] = s['promotion_id'].fillna(0).astype(int)                   # เติม 0 → int

# Date fix: แปลง → กรองปีเพี้ยน → เติม NaT → บวกเวลาสุ่ม 09:00-17:00
fix_dates = lambda col: (
    pd.to_datetime(col, errors='coerce', format='mixed')
    .pipe(lambda x: x.where(x.dt.year.between(2000, 2025)))
    .fillna(pd.Timestamp('2024-01-01')).dt.normalize()
    + pd.to_timedelta(np.random.randint(9*3600, 17*3600+1, size=len(col)), unit='s')
)

c['join_date'], c['last_purchase_date'], s['date'] = fix_dates(c['join_date']), fix_dates(c['last_purchase_date']), fix_dates(s['date'])

c.to_csv('customers_cleaned_short.csv', index=False)          # index=False → ไม่เอาเลขแถวลงไฟล์
s.to_csv('sales_transactions_cleaned_short.csv', index=False)
# จุดสังเกต: c['age'].isna().sum()==0, s['promotion_id'].dtype==int64